# LAB 08 - TravelOps
## Notebook: 00_seed_raw_data

Purpose:
Seeds selected `samples.wanderbricks` source tables into the Terraform-owned raw landing Volume.

Business purpose:
Creates repeatable raw travel booking data so CI/CD can prove the same application promotes from DEV to PROD.

Technical purpose:
Verifies the DAB target schema and Terraform-owned raw Volume, then overwrites deterministic Parquet folders used by Auto Loader.

Inputs:
- samples.wanderbricks.bookings
- samples.wanderbricks.booking_updates
- samples.wanderbricks.payments
- samples.wanderbricks.users
- samples.wanderbricks.properties
- samples.wanderbricks.reviews
- samples.wanderbricks.destinations

Outputs:
- /Volumes/<raw_volume_catalog>/<raw_volume_schema>/<raw_volume>/raw/<source_table>/

Tables/files affected:
Only raw Parquet folders are overwritten. Schema and Volume metadata are owned by Terraform.

Environment variables/widgets used:
`target_catalog`, `target_schema`, `raw_volume_catalog`, `raw_volume_schema`, `raw_volume_name`, `raw_volume_type`, `raw_volume_storage_location`, `source_catalog`, `source_schema`, `seed_limit`.

Creates/modifies data:
Yes. It writes deterministic raw landing files and is safe to rerun.

Dependencies/prerequisites:
The DAB target schema must exist, Terraform must have already created the raw Volume, and the deployer must have read access to `samples.wanderbricks`.

Expected result:
Each configured source table has one overwritten raw Parquet folder in the target Volume.

Failure behavior:
The notebook fails fast when required parameters are missing, when the Terraform-owned raw Volume is absent, or when source/target privileges are insufficient.

Environment classification:
Safe for personal_dev and personal_prod runs. Azure PROD job execution is intentionally deferred until explicitly authorized.


### Step 1 - Resolve target configuration

This cell reads Databricks widgets supplied by the Bundle job. It validates required values before any write occurs so configuration errors fail before partial data is created.

In [ ]:
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("target_schema", "")
dbutils.widgets.text("raw_volume_catalog", "")
dbutils.widgets.text("raw_volume_schema", "")
dbutils.widgets.text("raw_volume_name", "lab08_dev_travelops_raw")
dbutils.widgets.text("raw_volume_type", "MANAGED")
dbutils.widgets.text("raw_volume_storage_location", "")
dbutils.widgets.text("source_catalog", "samples")
dbutils.widgets.text("source_schema", "wanderbricks")
dbutils.widgets.text("seed_limit", "25000")

target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
raw_volume_catalog = dbutils.widgets.get("raw_volume_catalog")
raw_volume_schema = dbutils.widgets.get("raw_volume_schema")
raw_volume_name = dbutils.widgets.get("raw_volume_name")
raw_volume_type = dbutils.widgets.get("raw_volume_type").upper()
raw_volume_storage_location = dbutils.widgets.get("raw_volume_storage_location")
source_catalog = dbutils.widgets.get("source_catalog")
source_schema = dbutils.widgets.get("source_schema")
seed_limit = int(dbutils.widgets.get("seed_limit"))

required = {
    "target_catalog": target_catalog,
    "target_schema": target_schema,
    "raw_volume_catalog": raw_volume_catalog,
    "raw_volume_schema": raw_volume_schema,
    "raw_volume_name": raw_volume_name,
    "source_catalog": source_catalog,
    "source_schema": source_schema,
}
missing = [name for name, value in required.items() if not value]
if missing:
    raise ValueError(f"Missing required widgets: {missing}")
if raw_volume_type == "EXTERNAL" and not raw_volume_storage_location:
    raise ValueError("External raw volume requires raw_volume_storage_location")


### Step 2 - Verify target schema and Terraform-owned raw Volume

This cell performs read-only metadata checks. It intentionally does not create or alter the raw Volume schema or raw Volume because Terraform owns that layer; the application target schema is checked separately.

In [ ]:
target_schema_exists = spark.sql(f"SHOW SCHEMAS IN `{target_catalog}` LIKE '{target_schema}'").count() == 1
if not target_schema_exists:
    raise ValueError(f"Required DAB target schema does not exist: {target_catalog}.{target_schema}")

raw_schema_exists = spark.sql(f"SHOW SCHEMAS IN `{raw_volume_catalog}` LIKE '{raw_volume_schema}'").count() == 1
if not raw_schema_exists:
    raise ValueError(f"Required Terraform-referenced raw schema does not exist: {raw_volume_catalog}.{raw_volume_schema}")

volume_rows = spark.sql(f"SHOW VOLUMES IN `{raw_volume_catalog}`.`{raw_volume_schema}` LIKE '{raw_volume_name}'").collect()
if len(volume_rows) != 1:
    raise ValueError(f"Required Terraform-owned raw Volume is missing: {raw_volume_catalog}.{raw_volume_schema}.{raw_volume_name}")

print(f"Verified DAB target schema: {target_catalog}.{target_schema}")
print(f"Verified Terraform-owned raw Volume: {raw_volume_catalog}.{raw_volume_schema}.{raw_volume_name}")


### Step 3 - Seed deterministic raw Parquet folders

This cell reads the discovered Wanderbricks tables and overwrites deterministic target folders. Overwrite mode prevents uncontrolled duplicate files across reruns while keeping the raw contract file-based for Auto Loader.

In [ ]:
source_tables = [
    "bookings",
    "booking_updates",
    "payments",
    "users",
    "properties",
    "reviews",
    "destinations",
]

seed_summary = []
for table_name in source_tables:
    source_table = f"`{source_catalog}`.`{source_schema}`.`{table_name}`"
    target_path = f"/Volumes/{raw_volume_catalog}/{raw_volume_schema}/{raw_volume_name}/raw/{table_name}"
    df = spark.table(source_table).orderBy(*spark.table(source_table).columns[:1]).limit(seed_limit)
    row_count = df.count()
    df.write.mode("overwrite").format("parquet").save(target_path)
    seed_summary.append((table_name, row_count, target_path))

display(spark.createDataFrame(seed_summary, "table_name STRING, row_count LONG, target_path STRING"))
